# ADR Evaluation (clean, fast, consistent)

This notebook is a simpler replacement for `ADR_Eval.ipynb`.

It does:
1) **Rollout evaluation** (absolute + relative errors), batched.
2) **Intrinsic metrics** along trajectories:
   - Dynamics Jacobian singular values (∂f/∂z)
   - Decoder gain along trajectories (approx spectral norm of J_dec)
   - Latent error (z_pred vs z_true = enc(u_true))
3) **Paired comparisons** between methods (same windows).

**Consistency rules**:
- `umin/umax` are computed on **training μ and training-time** (coarse) and reused everywhere.
- Evaluation defaults to **fine** (`u_fine`, `t_fine`) but can be switched.

In [1]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.dataset import norm_u, inv_norm_u, H5Snapshots
from src.node import integrate_latent_eval

In [2]:
experiments = {
    "vanilla": {
        "lam_reg": 0
    },
    # "stoch_iso": {
    #     "lam_reg": 0.1
    # },    
    # "operator_norm": {
    #     "lam_reg": 0.1
    # },
    "curvature": {
        "lam_reg": 1.0,
    },
    "stiefel": {
        "lam_reg": 0
    }
}

In [3]:
# -----------------
# USER CONFIG
# -----------------
H5_PATH    = "adr_full.h5" 
SPLIT      = "extrap"           # "train" | "interp" | "extrap"
EVAL_FIELD = "u_fine"           # "u_fine" (recommended) or "u_coarse"
RK_METHOD  = "rk2"              # "rk2" or "rk4"

HORIZON    = 320                # in indices of EVAL_FIELD grid
N_MU       = 100
N_STARTS   = 10
BATCH      = 32

V          = 1                 # AE version
SELECTION  = "best_swa"        # ode_{selection}.pt   best_swa | target_swa | last_swa
METHODS    = [x for x in experiments.keys()]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32

# intrinsic metric cost controls
INTR_STEPS = 6
INTR_MAX_WINDOWS = 80
DEC_GAIN_ITERS = 5

SEED = 0
rng = np.random.default_rng(SEED)

print("DEVICE:", DEVICE)

DEVICE: cuda


In [4]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import copy

# --------- CONFIG ----------
TOP_K = 3

# If you want to restrict versions/runs/methods, set these; otherwise we auto-discover.
AE_VERSIONS = [1]
NODE_RUNS   =  range(1,10+1) # range(1,16+1)

# "Robust worst-best" target selection:
# q = 1.0  -> max(best_vals)  (strict "worst best")
# q = 0.9  -> 90th percentile (more robust to one outlier)
TARGET_Q = 1.0

OUTPUT_ROOT = Path("output/NODE_selected")
MAP_LOCATION = "cuda"


In [5]:
def parse_method_version_run(stats_path: Path):
    # stats_path: output/NODE_selected/{method}_v{version}/run_{run}/stats.json
    run_dir = stats_path.parent
    method_v = run_dir.parent.name          # "{method}_v{version}"
    run_name = run_dir.name                 # "run_{run}"

    if "_v" not in method_v:
        raise ValueError(f"Cannot parse version from: {method_v}")
    method, v_str = method_v.rsplit("_v", 1)
    version = int(v_str)

    if not run_name.startswith("run_"):
        raise ValueError(f"Cannot parse run from: {run_name}")
    run = int(run_name.split("_", 1)[1])

    return method, version, run, run_dir

# Discover all stats.json files
stats_files = sorted(OUTPUT_ROOT.glob("*_v*/run_*/stats.json"))
print("Found stats.json:", len(stats_files))

records = []
stats_map = {}  # (version, method, run) -> dict(stats) + run_dir

for p in stats_files:
    method, version, run, run_dir = parse_method_version_run(p)

    if AE_VERSIONS is not None and version not in AE_VERSIONS:
        continue
    if NODE_RUNS is not None and run not in NODE_RUNS:
        continue

    with open(p, "r") as f:
        st = json.load(f)

    # basic sanity
    if "mse_val" not in st or "mse_train" not in st:
        print(f"[skip] missing mse_train/mse_val in {p}")
        continue

    stats_map[(version, method, run)] = {"stats": st, "run_dir": run_dir}

print("Loaded runs:", len(stats_map))
# list(stats_map.keys())


Found stats.json: 30
Loaded runs: 30


In [6]:
best_vals = []
for (version, method, run), obj in stats_map.items():
    val = np.asarray(obj["stats"]["mse_val"], dtype=float)
    best_vals.append(val.min())

best_vals = np.asarray(best_vals, dtype=float)
if len(best_vals) == 0:
    raise RuntimeError("No stats loaded; check OUTPUT_ROOT or filters.")

target_val = float(np.quantile(best_vals, TARGET_Q))
print(f"Target val MSE: (q={TARGET_Q}): {target_val:.6e}")
print(f"Best-val range: {float(best_vals.min()):.6e} to {float(best_vals.max()):.6e}")


Target val MSE: (q=1.0): 3.243552e-03
Best-val range: 5.007822e-04 to 3.243552e-03


In [7]:
def first_epoch_runningbest_leq(val_mse: np.ndarray, target: float):
    """
    Return 1-based epoch where running best first reaches <= target.
    If never reaches, return None.
    """
    rb = np.minimum.accumulate(val_mse)
    hits = np.where(rb <= target)[0]
    if len(hits) == 0:
        return None
    return int(hits[0] + 1)

def trailing_epochs(ep: int, k: int):
    """Return [max(1, ep-k+1), ..., ep] (1-based)."""
    start = max(1, ep - k + 1)
    return list(range(start, ep + 1))

def load_ckpt_model(run_dir: Path, epoch: int):
    ckpt_path = run_dir / f"ode_{epoch}.pt"
    if not ckpt_path.exists():
        return None, ckpt_path
    model = torch.load(ckpt_path, map_location=MAP_LOCATION, weights_only=False)
    model.eval()
    return model, ckpt_path

def swa_average_checkpoints(run_dir: Path, epochs: list[int], save_path: Path):
    """
    Uniform average of floating tensors over checkpoints in `epochs`.
    Non-floating tensors are copied from the LAST loaded checkpoint.
    Saves a torch model at save_path.
    Returns: (used_epochs, save_path) or ([], None) if nothing loaded.
    """
    used = []
    sum_sd = None
    last_sd = None
    template_model = None

    for ep in epochs:
        model, ckpt_path = load_ckpt_model(run_dir, ep)
        if model is None:
            print(f"[warn] missing {ckpt_path}")
            continue

        sd = model.state_dict()
        if sum_sd is None:
            template_model = copy.deepcopy(model)
            sum_sd = {k: v.detach().clone().to(MAP_LOCATION) for k, v in sd.items()}
        else:
            for k in sum_sd:
                if torch.is_floating_point(sum_sd[k]) or torch.is_complex(sum_sd[k]):
                    sum_sd[k].add_(sd[k].detach().to(MAP_LOCATION, dtype=sum_sd[k].dtype))
                else:
                    # keep placeholder; we'll overwrite from last checkpoint below
                    pass

        last_sd = {k: v.detach().to(MAP_LOCATION) for k, v in sd.items()}
        used.append(ep)

    if not used:
        print(f"[skip] no checkpoints found for {run_dir}")
        return [], None

    # divide floating tensors
    n = float(len(used))
    for k in sum_sd:
        if torch.is_floating_point(sum_sd[k]) or torch.is_complex(sum_sd[k]):
            sum_sd[k].div_(n)
        else:
            # copy non-floating from last checkpoint
            sum_sd[k] = last_sd[k].clone()

    template_model.load_state_dict(sum_sd, strict=True)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(template_model, save_path)
    return used, save_path


In [8]:
rows = []

for (version, method, run), obj in sorted(stats_map.items()):
    st = obj["stats"]
    run_dir: Path = obj["run_dir"]

    train = np.asarray(st["mse_train"], dtype=float)
    val   = np.asarray(st["mse_val"], dtype=float)
    n_epochs = len(val)

    # epochs (1-based)
    best_train_ep = int(np.argmin(train) + 1)
    best_val_ep   = int(np.argmin(val) + 1)

    # "target reached" epoch using running best
    target_ep = first_epoch_runningbest_leq(val, target_val)
    target_reached = target_ep is not None
    if target_ep is None:
        target_ep = n_epochs  # fallback: use last epoch if never reached

    # trailing windows
    epochs_target = trailing_epochs(target_ep, TOP_K)
    epochs_best   = trailing_epochs(best_val_ep, TOP_K)
    epochs_last   = trailing_epochs(n_epochs, TOP_K)

    # save SWA models
    used_t, p_t = swa_average_checkpoints(run_dir, epochs_target, run_dir / "ode_target_swa.pt")
    used_b, p_b = swa_average_checkpoints(run_dir, epochs_best,   run_dir / "ode_best_swa.pt")
    used_l, p_l = swa_average_checkpoints(run_dir, epochs_last,   run_dir / "ode_last_swa.pt")

    rows.append({
        "version": version,
        "method": method,
        "run": run,
        "n_epochs": n_epochs,

        "best_train": float(train.min()),
        "best_val": float(val.min()),
        "best_train_ep": best_train_ep,
        "best_val_ep": best_val_ep,

        "target_val": float(target_val),
        "target_ep": int(target_ep),
        "target_reached": bool(target_reached),

        "std_train": float(train.std(ddof=0)),
        "std_val": float(val.std(ddof=0)),

        "target_swa_epochs": used_t,
        "best_swa_epochs": used_b,
        "last_swa_epochs": used_l,
    })

metadata_df = pd.DataFrame(rows).sort_values(["version", "method", "run"]).reset_index(drop=True)
metadata_df


,version,method,run,n_epochs,best_train,best_val,best_train_ep,best_val_ep,target_val,target_ep,target_reached,std_train,std_val,target_swa_epochs,best_swa_epochs,last_swa_epochs
0,1,curvature,1,50,0.001317,0.001636,50,49,0.003244,15,True,0.028123,0.026749,"[13, 14, 15]","[47, 48, 49]","[48, 49, 50]"
1,1,curvature,2,50,0.001442,0.001929,49,38,0.003244,16,True,0.028294,0.026026,"[14, 15, 16]","[36, 37, 38]","[48, 49, 50]"
2,1,curvature,3,50,0.001189,0.002041,48,50,0.003244,20,True,0.027920,0.027051,"[18, 19, 20]","[48, 49, 50]","[48, 49, 50]"
3,1,curvature,4,50,0.001222,0.001192,50,48,0.003244,13,True,0.027879,0.026246,"[11, 12, 13]","[46, 47, 48]","[48, 49, 50]"
4,1,curvature,5,50,0.001028,0.003244,49,31,0.003244,31,True,0.028870,0.026486,"[29, 30, 31]","[29, 30, 31]","[48, 49, 50]"
5,1,curvature,6,50,0.001179,0.001206,49,44,0.003244,16,True,0.028199,0.027389,"[14, 15, 16]","[42, 43, 44]","[48, 49, 50]"
6,1,curvature,7,50,0.001364,0.002621,50,44,0.003244,21,True,0.029325,0.027833,"[19, 20, 21]","[42, 43, 44]","[48, 49, 50]"
7,1,curvature,8,50,0.001079,0.001390,49,35,0.003244,15,True,0.028627,0.027257,"[13, 14, 15]","[33, 34, 35]","[48, 49, 50]"
8,1,curvature,9,50,0.000935,0.001068,50,48,0.003244,13,True,0.028512,0.027049,"[11, 12, 13]","[46, 47, 48]","[48, 49, 50]"
9,1,curvature,10,50,0.001166,0.002054,49,49,0.003244,27,True,0.028353,0.026748,"[25, 26, 27]","[47, 48, 49]","[48, 49, 50]"


In [9]:
# Summary across runs: mean/std of key quantities per (version, method)
agg_cols = [
    "best_train", "best_val",
    "best_train_ep", "best_val_ep",
    "target_ep", "std_train", "std_val"
]

summary_df = (
    metadata_df
    .groupby(["version", "method"], as_index=False)[agg_cols]
    .agg(["mean", "std", "count"])
)

# flatten MultiIndex columns
summary_df.columns = ["version", "method"] + [f"{c0}_{c1}" for (c0, c1) in summary_df.columns[2:]]
summary_df = summary_df.sort_values(["version", "best_val_mean"]).reset_index(drop=True)
summary_df


,version,method,best_train_mean,best_train_std,best_train_count,best_val_mean,best_val_std,best_val_count,best_train_ep_mean,best_train_ep_std,...,best_val_ep_count,target_ep_mean,target_ep_std,target_ep_count,std_train_mean,std_train_std,std_train_count,std_val_mean,std_val_std,std_val_count
0,1,stiefel,0.000580,0.000061,10,0.000915,0.000368,10,49.8,0.421637,...,10,12.0,3.681787,10,0.023018,0.000766,10,0.013427,0.002158,10
1,1,vanilla,0.000663,0.000071,10,0.001013,0.000368,10,49.8,0.421637,...,10,11.8,3.224903,10,0.023870,0.000585,10,0.016563,0.002431,10
2,1,curvature,0.001192,0.000155,10,0.001838,0.000694,10,49.3,0.674949,...,10,18.7,6.092801,10,0.028410,0.000444,10,0.026883,0.000547,10


In [10]:
# -----------------
# Helpers: time grids + split segments from H5
# -----------------
def read_time_and_kind(f: h5py.File, field: str):
    if "coarse" in field:
        return f["t_coarse"][...], "coarse"
    return f["t_fine"][...], "fine"

def load_segment(f: h5py.File, kind: str, split: str):
    # Prefer explicit split datasets from the paper-aligned generator
    if "splits" in f:
        if kind == "fine":
            key = {"train":"splits/fine_ids_train",
                   "interp":"splits/fine_ids_interp",
                   "extrap":"splits/fine_ids_extrap"}[split]
        else:
            key = {"train":"splits/coarse_ids_train",
                   "interp":"splits/coarse_ids_val_loop",
                   "extrap":"splits/coarse_ids_val_loop"}[split]
        ids = np.asarray(f[key][...], dtype=np.int64)
        return int(ids[0]), int(ids[-1])

    # Fallback: infer from attrs iT1_*, iT2_*
    if kind == "fine":
        iT1 = int(f.attrs["iT1_fine"]); iT2 = int(f.attrs["iT2_fine"]); Nt = f["t_fine"].shape[0]
    else:
        iT1 = int(f.attrs["iT1_coarse"]); iT2 = int(f.attrs["iT2_coarse"]); Nt = f["t_coarse"].shape[0]

    if split == "train":  return 0, iT1
    if split == "interp": return iT1 + 1, iT2
    if split == "extrap": return iT2 + 1, Nt - 1
    raise ValueError(split)

with h5py.File(H5_PATH, "r") as f:
    t_eval_np, kind = read_time_and_kind(f, EVAL_FIELD)
    seg0, seg1 = load_segment(f, kind, SPLIT)
    U = f[f"{SPLIT}/{EVAL_FIELD}"]
    Nmu_all, Nt_all, H, W = U.shape

print("EVAL_FIELD:", EVAL_FIELD, "kind:", kind, "U:", (Nmu_all, Nt_all, H, W))
print("Segment:", SPLIT, "t ids:", (seg0, seg1), "len:", (seg1 - seg0 + 1))

EVAL_FIELD: u_fine kind: fine U: (200, 1001, 32, 32)
Segment: extrap t ids: (501, 1000) len: 500


In [11]:
# -----------------
# Normalization: compute umin/umax on (train μ) x (train-time) using u_coarse.
# Cached to norm_cache.json.
# -----------------
def get_or_compute_norm(h5_path: str, cache_path: str = "norm_cache.json"):
    if os.path.exists(cache_path):
        with open(cache_path, "r") as g:
            d = json.load(g)
        if d.get("h5_path") == os.path.abspath(h5_path):
            return float(d["umin"]), float(d["umax"])

    with h5py.File(h5_path, "r") as f:
        train_idx = np.asarray(f["train_idx"][...], dtype=np.int64)
        if "splits" in f and "coarse_ids_train" in f["splits"]:
            train_t_ids = np.asarray(f["splits/coarse_ids_train"][...], dtype=np.int64)
        else:
            iT1 = int(f.attrs["iT1_coarse"])
            train_t_ids = np.arange(0, iT1 + 1, dtype=np.int64)

    ds = H5Snapshots(h5_path, split="train", field="u_coarse", mu_indices=train_idx, return_mu=False)
    umin, umax = ds.compute_train_minmax(field="u_coarse", mu_ids=train_idx, t_ids=train_t_ids, split="train")

    with open(cache_path, "w") as g:
        json.dump({"h5_path": os.path.abspath(h5_path), "umin": float(umin), "umax": float(umax)}, g)

    return float(umin), float(umax)

umin, umax = get_or_compute_norm(H5_PATH)
print("umin/umax:", umin, umax)

umin/umax: 0.0 3.20582914352417


In [12]:
# -----------------
# Sample evaluation windows (paired across methods)
# -----------------
def sample_windows(h5_path: str, split: str, field: str, horizon: int, n_mu: int, n_starts: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    with h5py.File(h5_path, "r") as f:
        U = f[f"{split}/{field}"]
        Nmu, Nt, *_ = U.shape
        t_np, kind = read_time_and_kind(f, field)
        seg0, seg1 = load_segment(f, kind, split)

    max_k0 = seg1 - horizon
    if max_k0 < seg0:
        raise ValueError(f"Segment too short for horizon={horizon}: seg=[{seg0},{seg1}]")

    mu_ids = rng.choice(np.arange(Nmu), size=min(n_mu, Nmu), replace=True)
    windows = []
    for mu in mu_ids:
        k0s = rng.integers(low=seg0, high=max_k0 + 1, size=n_starts)
        for k0 in k0s:
            windows.append((int(mu), int(k0)))
    return windows

windows = sample_windows(H5_PATH, SPLIT, EVAL_FIELD, HORIZON, N_MU, N_STARTS, seed=SEED)
print("Num windows:", len(windows), "example:", windows[:5])

Num windows: 1000 example: [(170, 573), (170, 642), (170, 557), (170, 544), (170, 643)]


In [13]:
# -----------------
# Load models
# -----------------
def load_models(methods, V: int, run: int, selection: str, device: str):
    out = {}
    for m in methods:
        enc = torch.load(f"output/AE_v{V}/{m}/enc_best.pt", map_location=device, weights_only=False)
        dec = torch.load(f"output/AE_v{V}/{m}/dec_best.pt", map_location=device, weights_only=False)
        ode = torch.load(f"output/NODE_selected/{m}_v{V}/run_{run}/ode_{selection}.pt", map_location=device, weights_only=False)
        enc.eval(); dec.eval(); ode.eval()
        out[m] = (enc.to(device), dec.to(device), ode.to(device))
    return out



In [14]:
# -----------------
# Batched rollout + error metrics
# -----------------
def get_mu_tensor(mu_all_np, mu_ids, device):
    # mu_all_np: (Nmu, 3) float32 numpy
    return torch.as_tensor(mu_all_np[mu_ids], device=device, dtype=DTYPE)

def gather_u_windows(f: h5py.File, split: str, field: str, batch_windows, horizon: int):
    U = f[f"{split}/{field}"]
    out = []
    for (mu, k0) in batch_windows:
        out.append(np.asarray(U[mu, k0:k0+horizon+1], dtype=np.float32))  # (L+1,H,W)
    return np.stack(out, axis=0)  # (B,L+1,H,W)

def build_time_grid(t_np: np.ndarray, k0s: np.ndarray, horizon: int, device: str):
    """
    Exact absolute-time grid per window, required because the ODE uses absolute-time encoding.
    Returns t_grid: (B, L+1) torch float32.
    """
    offsets = np.arange(horizon + 1, dtype=np.int64)[None, :]  # (1,L+1)
    idx = k0s[:, None] + offsets                                # (B,L+1)
    t_grid_np = t_np[idx]                                       # (B,L+1)
    return torch.as_tensor(t_grid_np, device=device, dtype=DTYPE)

@torch.no_grad()
def rollout(enc, dec, ode, u_win_np, t_grid, mu_t, umin, umax, rk_method="rk2", drop_t0=True):
    """
    u_win_np: (B,L+1,H,W) physical
    t_grid:   (B,L+1) absolute time
    mu_t:     (B,nmu)
    """
    B, Lp1, H, W = u_win_np.shape
    device = mu_t.device

    u_true = torch.as_tensor(u_win_np, device=device, dtype=DTYPE).unsqueeze(2)  # (B,L+1,1,H,W)
    u0 = u_true[:, 0]  # (B,1,H,W)

    u0_n = norm_u(u0, umin, umax)
    z0 = enc(u0_n)

    z_pred = integrate_latent_eval(ode, z0, t_grid, mu_t, method=rk_method)   # (B,L+1,1,4,4)
    u_hat_n = dec(z_pred.reshape(-1, *z_pred.shape[2:])).reshape(B, Lp1, 1, H, W)
    u_hat = inv_norm_u(u_hat_n, umin, umax)

    # errors (physical space)
    diff = (u_hat - u_true).reshape(B, Lp1, -1)
    tru  = (u_true).reshape(B, Lp1, -1)

    num = torch.linalg.vector_norm(diff, dim=-1)  # (B,L+1)
    den = torch.linalg.vector_norm(tru,  dim=-1)  # (B,L+1)

    if drop_t0:
        num = num[:, 1:]
        den = den[:, 1:]
        diff = diff[:, 1:]

    rel = num / (den + 1e-8)
    mse = diff.pow(2).mean(dim=-1)

    metrics = {
        "abs_mean": num.mean(dim=1),
        "abs_max":  num.max(dim=1).values,
        "abs_fin":  num[:, -1],
        "rel_mean": rel.mean(dim=1),
        "rel_max":  rel.max(dim=1).values,
        "rel_fin":  rel[:, -1],
        "mse_fin":  mse[:, -1],
    }
    return metrics, z_pred, u_hat, u_true

In [15]:
# -----------------
# Intrinsic metrics
# -----------------
def dyn_jac_svals(ode, z, t_scalar, mu_row):
    with torch.enable_grad():
        z_flat = z.reshape(-1).detach().requires_grad_(True)

        def f_flat(z_flat_):
            zz = z_flat_.view(1, 1, 4, 4)
            out = ode(t_scalar.view(1), zz, mu_row.view(1, -1))
            return out.reshape(-1)

        J = torch.autograd.functional.jacobian(f_flat, z_flat, create_graph=False)
        return torch.linalg.svdvals(J).detach()


def decoder_gain_power(dec, z, n_iter=5):
    """Approx spectral norm of J_dec at z via power iteration on J^T J."""
    with torch.enable_grad():
        v = torch.randn_like(z)
        v = v / (v.flatten().norm() + 1e-12)

        for _ in range(n_iter):
            z0 = z.detach().requires_grad_(True)

            # y has a grad graph wrt z0
            y = dec(z0).flatten()

            # Jv as a *constant* vector (no need for create_graph=True)
            _, jv = torch.autograd.functional.jvp(
                lambda zz: dec(zz).flatten(),
                (z0,),
                (v,),
                create_graph=False
            )

            # grad wrt z0 gives J^T (Jv)
            dot = (y * jv.detach()).sum()
            jt_jv = torch.autograd.grad(dot, z0, retain_graph=False, create_graph=False)[0]

            v = jt_jv.detach()
            v = v / (v.flatten().norm() + 1e-12)

        # final sigma estimate = ||J v||
        z0 = z.detach().requires_grad_(True)
        _, jv = torch.autograd.functional.jvp(
            lambda zz: dec(zz).flatten(),
            (z0,),
            (v,),
            create_graph=False
        )
        return jv.norm().detach()


@torch.no_grad()
def latent_truth(enc, u_true, umin, umax):
    B, Lp1 = u_true.shape[0], u_true.shape[1]
    u_n = norm_u(u_true.reshape(-1, *u_true.shape[2:]), umin, umax)
    z_true = enc(u_n).reshape(B, Lp1, 1, 4, 4)
    return z_true

def intrinsic_for_batch(enc, dec, ode, z_pred, u_true, t_grid, mu_t, steps_idx, dec_iters=5):
    rows = []
    z_true = latent_truth(enc, u_true, umin, umax)

    B = z_pred.shape[0]
    for b in range(B):
        mu_row = mu_t[b]
        for sidx in steps_idx:
            z = z_pred[b, sidx:sidx+1]           # (1,1,4,4)
            t_scalar = t_grid[b, sidx]

            svals = dyn_jac_svals(ode, z, t_scalar, mu_row)
            gain  = decoder_gain_power(dec, z, n_iter=dec_iters)
            zerr  = (z_pred[b, sidx] - z_true[b, sidx]).reshape(-1).norm()

            rows.append({
                "b": b,
                "step": int(sidx),
                "t": float(t_scalar.detach().cpu()),
                "dyn_sigma_max": float(svals.max().cpu()),
                "dyn_sigma_min": float(svals.min().cpu()),
                "dyn_cond": float((svals.max()/(svals.min()+1e-12)).cpu()),
                "dec_gain": float(gain.cpu()),
                "latent_err": float(zerr.detach().cpu()),
            })
    return pd.DataFrame(rows)

In [16]:
# -----------------
# Evaluate all methods (paired windows)
# -----------------
def eval_all(models, windows, split, field, horizon, batch_size):
    all_rows = []
    intr_rows = []

    steps_idx = np.linspace(0, horizon, num=INTR_STEPS).round().astype(int).tolist()
    intr_windows = set(windows[:min(INTR_MAX_WINDOWS, len(windows))])

    with h5py.File(H5_PATH, "r") as f:
        t_np, _ = read_time_and_kind(f, field)

        MU_ALL = None
        mu_key = f"{split}/mu"
        if mu_key in f:
            MU_ALL = np.asarray(f[mu_key][...], dtype=np.float32)

        for method, (enc, dec, ode) in models.items():
            print("Evaluating:", method)

            for i0 in range(0, len(windows), batch_size):
                batch_windows = windows[i0:i0+batch_size]
                mu_ids = np.asarray([w[0] for w in batch_windows], dtype=np.int64)
                k0s    = np.asarray([w[1] for w in batch_windows], dtype=np.int64)

                u_win_np = gather_u_windows(f, split, field, batch_windows, horizon)
                t_grid = build_time_grid(t_np, k0s, horizon, DEVICE)

                mu_t = (
                    get_mu_tensor(MU_ALL, mu_ids, DEVICE)
                    if MU_ALL is not None
                    else torch.zeros((len(mu_ids), 0), device=DEVICE, dtype=DTYPE)
                )

                metrics, z_pred, u_hat, u_true = rollout(
                    enc, dec, ode, u_win_np, t_grid, mu_t, umin, umax, rk_method=RK_METHOD
                )

                for j, (mu_id, k0) in enumerate(batch_windows):
                    all_rows.append({
                        "method": method,
                        "split": split,
                        "field": field,
                        "horizon": horizon,
                        "mu_id": int(mu_id),
                        "k0": int(k0),
                        "abs_mean": float(metrics["abs_mean"][j].cpu()),
                        "abs_max":  float(metrics["abs_max"][j].cpu()),
                        "abs_fin":  float(metrics["abs_fin"][j].cpu()),
                        "rel_mean": float(metrics["rel_mean"][j].cpu()),
                        "rel_max":  float(metrics["rel_max"][j].cpu()),
                        "rel_fin":  float(metrics["rel_fin"][j].cpu()),
                        "mse_fin":  float(metrics["mse_fin"][j].cpu()),
                    })

                if any((w in intr_windows) for w in batch_windows):
                    keep = [idx for idx,w in enumerate(batch_windows) if w in intr_windows]
                    if keep:
                        df_intr = intrinsic_for_batch(
                            enc, dec, ode,
                            z_pred[keep], u_true[keep], t_grid[keep], mu_t[keep],
                            steps_idx=steps_idx, dec_iters=DEC_GAIN_ITERS
                        )
                        df_intr["method"] = method
                        df_intr["split"] = split
                        df_intr["field"] = field
                        df_intr["horizon"] = horizon
                        df_intr["mu_id"] = df_intr["b"].map(lambda bb: int(batch_windows[keep[bb]][0]))
                        df_intr["k0"]    = df_intr["b"].map(lambda bb: int(batch_windows[keep[bb]][1]))
                        df_intr.drop(columns=["b"], inplace=True)
                        intr_rows.append(df_intr)

    df_roll = pd.DataFrame(all_rows)
    df_intr = pd.concat(intr_rows, ignore_index=True) if intr_rows else pd.DataFrame()
    return df_roll, df_intr


# -----------------
# Summary tables
# -----------------
def summarize(df, cols):
    g = df.groupby("method")[cols]
    out = g.agg(["mean","std","count"])
    for c in cols:
        out[(c,"se")] = out[(c,"std")] / np.sqrt(out[(c,"count")].clip(lower=1))
    return out.sort_values((cols[0],"mean"))


# -----------------
# Paired comparisons vs vanilla
# -----------------
from scipy.stats import wilcoxon

def paired_vs_baseline(df, metric: str, baseline: str = "vanilla"):
    keys = ["mu_id","k0"]
    wide = df.pivot_table(index=keys, columns="method", values=metric, aggfunc="first")
    if baseline not in wide.columns:
        raise ValueError(f"baseline {baseline} not present in {list(wide.columns)}")
    deltas = wide.sub(wide[baseline], axis=0).drop(columns=[baseline], errors="ignore")

    rows=[]
    for m in deltas.columns:
        x = deltas[m].dropna().to_numpy()
        if len(x) >= 5:
            p_less = wilcoxon(x, alternative="less").pvalue     # delta < 0 is better
            p_two  = wilcoxon(x, alternative="two-sided").pvalue
        else:
            p_less = p_two = np.nan
        rows.append({
            "metric": metric,
            "method": m,
            "n": len(x),
            "delta_mean": float(np.mean(x)) if len(x) else np.nan,
            "delta_median": float(np.median(x)) if len(x) else np.nan,
            "p_less": p_less,
            "p_two_sided": p_two,
        })
    return pd.DataFrame(rows).sort_values("p_less")

    
for seed in NODE_RUNS:
    print(f"RUN {seed} [{SELECTION}] ----------------------------------------")

    models = load_models(METHODS, V, seed, SELECTION, DEVICE)
    print("Loaded:", list(models.keys()))

    df_roll, df_intr = eval_all(models, windows, SPLIT, EVAL_FIELD, HORIZON, BATCH)
    print(df_roll.head())
    print("roll rows:", len(df_roll), "intr rows:", len(df_intr))

    summary = summarize(df_roll, ["rel_mean","rel_max","rel_fin","abs_fin","mse_fin"])
    print(f"[{seed}] summary:")
    print(summary)

    
    paired = paired_vs_baseline(df_roll, "rel_mean")
    print(f"{seed} [{SELECTION}] paired:")
    print(paired)

    os.makedirs(f"selected_stats/{SELECTION}", exist_ok=True)

    df_roll.to_csv(f"selected_stats/{SELECTION}/df_roll_H{HORIZON}_v{V}_{seed}.csv")
    df_intr.to_csv(f"selected_stats/{SELECTION}/df_intr_H{HORIZON}{V}_{seed}.csv")  

    summary.to_csv(f"selected_stats/{SELECTION}/summary_H{HORIZON}_v{V}_{seed}.csv")

    paired.to_csv(f"selected_stats/{SELECTION}/paired_H{HORIZON}_v{V}_{seed}.csv")

    print("----------------------------------------\n")

RUN 1 [best_swa] ----------------------------------------
Loaded: ['vanilla', 'curvature', 'stiefel']
Evaluating: vanilla
Evaluating: curvature
Evaluating: stiefel
    method   split   field  horizon  mu_id   k0   abs_mean    abs_max  \
0  vanilla  extrap  u_fine      320    170  573  10.183086  14.050152   
1  vanilla  extrap  u_fine      320    170  642   8.470984  11.097225   
2  vanilla  extrap  u_fine      320    170  557   9.391273  12.560951   
3  vanilla  extrap  u_fine      320    170  544   9.586441  12.528597   
4  vanilla  extrap  u_fine      320    170  643   8.430623  11.011642   

     abs_fin  rel_mean   rel_max   rel_fin   mse_fin  
0  11.234538  0.218865  0.312303  0.279716  0.123257  
1   9.497136  0.200872  0.282970  0.269911  0.088082  
2  11.717005  0.201870  0.295403  0.228988  0.134071  
3  12.172402  0.207862  0.298664  0.216991  0.144695  
4   9.578528  0.200224  0.281144  0.271394  0.089598  
roll rows: 3000 intr rows: 1440
[1] summary:
           rel_mean   

In [17]:
df_intr = pd.DataFrame()
df_roll = pd.DataFrame()


for seed in range(1,4+1):
    df_i = pd.read_csv(f"selected_stats/{SELECTION}/df_intr_H{HORIZON}{V}_{seed}.csv") 
    df_r = pd.read_csv(f"selected_stats/{SELECTION}/df_roll_H{HORIZON}_v{V}_{seed}.csv")
    
    df_intr = pd.concat([df_intr,df_i])
    df_roll = pd.concat([df_roll,df_r])



In [18]:
df_intr.head(10)

,Unnamed: 0,step,t,dyn_sigma_max,dyn_sigma_min,dyn_cond,dec_gain,latent_err,method,split,field,horizon,mu_id,k0
0,0,0,18.001326,3.131551,0.015215,205.813660,1.989995,0.000236,vanilla,extrap,u_fine,320,170,573
1,1,64,20.011946,4.371011,0.028069,155.725143,2.116656,3.419177,vanilla,extrap,u_fine,320,170,573
2,2,128,22.022564,3.916076,0.002590,1512.199951,2.035167,4.330840,vanilla,extrap,u_fine,320,170,573
3,3,192,24.033184,3.566241,0.018419,193.617630,2.036977,4.028521,vanilla,extrap,u_fine,320,170,573
4,4,256,26.043802,5.995787,0.044726,134.054733,2.110307,4.055388,vanilla,extrap,u_fine,320,170,573
5,5,320,28.054422,3.419059,0.012427,275.127197,2.041184,4.731530,vanilla,extrap,u_fine,320,170,573
6,6,0,20.169025,5.032559,0.019277,261.071014,2.101554,0.000361,vanilla,extrap,u_fine,320,170,642
7,7,64,22.179644,3.349366,0.027277,122.791969,1.985054,3.670673,vanilla,extrap,u_fine,320,170,642
8,8,128,24.190264,2.476441,0.000151,16393.171875,2.007617,3.404044,vanilla,extrap,u_fine,320,170,642
9,9,192,26.200882,4.848891,0.001197,4050.571045,2.109150,2.748628,vanilla,extrap,u_fine,320,170,642


In [19]:
# -----------------
# Intrinsic metric summaries
# -----------------
if len(df_intr):
    intr_summary = df_intr.groupby("method")[["dyn_sigma_max","dyn_cond","dec_gain","latent_err"]].agg(["mean","std","count"])
    intr_summary
else:
    print("No intrinsic rows computed.")

In [20]:
intr_summary

dyn_sigma_max                     dyn_cond                      \
                   mean       std count         mean           std count   
method                                                                     
curvature      6.152667  1.692872  1920  2687.340528  17872.374568  1920   
stiefel        3.570942  0.930460  1920  1521.006197  19701.863159  1920   
vanilla        3.941454  1.016101  1920  1303.542589   6332.973738  1920   

           dec_gain                 latent_err                  
               mean       std count       mean       std count  
method                                                          
curvature  0.631314  0.034497  1920  10.675307  7.770878  1920  
stiefel    3.254238  0.151593  1920   1.524477  1.032088  1920  
vanilla    2.011550  0.111872  1920   2.268761  1.553262  1920

In [21]:
# intr_summary.to_csv(f"stats/{SELECTION}/intr_summary_H{HORIZON}_v{V}_{RUN}.csv")

In [22]:
import numpy as np
import pandas as pd

df = df_intr.copy()
df = df.drop(columns=[c for c in ["Unnamed: 0"] if c in df.columns])

# Make sure numeric
num_cols = ["step","t","dyn_sigma_max","dyn_sigma_min","dyn_cond","dec_gain","latent_err"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Logs (clip avoids -inf if something hits 0)
eps = 1e-12
df["log10_dyn_cond"] = np.log10(np.clip(df["dyn_cond"].to_numpy(), eps, None))
df["log10_sigma_max"] = np.log10(np.clip(df["dyn_sigma_max"].to_numpy(), eps, None))
df["log10_sigma_min"] = np.log10(np.clip(df["dyn_sigma_min"].to_numpy(), eps, None))
df["log10_dec_gain"]  = np.log10(np.clip(df["dec_gain"].to_numpy(), eps, None))

# Tail / “catastrophe” rates (tune thresholds after looking at quantiles)
df["cond_gt_1e4"]  = df["dyn_cond"] > 1e4
df["cond_gt_1e5"]  = df["dyn_cond"] > 1e5
df["sigmax_gt_10"] = df["dyn_sigma_max"] > 10


In [23]:
qs = [0.5, 0.9, 0.95, 0.99, 0.999]

def qcols(s):
    q = s.quantile(qs)
    q.index = [f"q{int(p*1000):03d}" for p in qs]  # q500 q900 q950 q990 q999
    return q

summary_method = (
    df.groupby("method")
      .apply(lambda g: pd.concat([
          qcols(g["log10_dyn_cond"]).add_prefix("log10_dyn_cond_"),
          qcols(g["latent_err"]).add_prefix("latent_err_"),
          qcols(g["dyn_sigma_max"]).add_prefix("sigmax_"),
          g[["cond_gt_1e4","cond_gt_1e5","sigmax_gt_10"]].mean().add_prefix("rate_"),
      ]))
      .reset_index()
)

summary_method


,method,log10_dyn_cond_q500,log10_dyn_cond_q900,log10_dyn_cond_q950,log10_dyn_cond_q990,log10_dyn_cond_q999,latent_err_q500,latent_err_q900,latent_err_q950,latent_err_q990,latent_err_q999,sigmax_q500,sigmax_q900,sigmax_q950,sigmax_q990,sigmax_q999,rate_cond_gt_1e4,rate_cond_gt_1e5,rate_sigmax_gt_10
0,curvature,2.833461,3.568101,3.877212,4.452806,5.168141,9.690028,20.505213,25.004305,32.800313,46.749528,5.970553,8.367038,9.267195,10.963470,13.588239,0.036458,0.003125,0.025521
1,stiefel,2.338265,3.047197,3.377069,4.099803,5.454538,1.455820,2.879159,3.475291,4.221927,4.938071,3.500088,4.778947,5.273529,6.139217,7.005983,0.013542,0.001563,0.000000
2,vanilla,2.516453,3.280250,3.583036,4.224948,4.970881,2.158171,4.369825,4.943408,6.633346,8.258678,3.846237,5.265375,5.798549,6.963679,8.057507,0.016146,0.000521,0.000000


In [ ]:
run_keys = ["method","split","field","horizon","mu_id","k0"]

run_stats = (
    df.groupby(run_keys)
      .agg(
          logcond_med=("log10_dyn_cond","median"),
          logcond_p99=("log10_dyn_cond", lambda s: s.quantile(0.99)),
          cond_rate_1e5=("cond_gt_1e5","mean"),
          sigmax_med=("dyn_sigma_max","median"),
          laterr_med=("latent_err","median"),
      )
      .reset_index()
)

# Aggregate across runs (now each run is one row)
method_run_summary = (
    run_stats.groupby(["method","split","field","horizon"])
             .agg(["mean","std","median"])
)

method_run_summary


In [ ]:
time_profile = (
    df.groupby(["method","split","field","horizon","step"])
      .agg(
          logcond_med=("log10_dyn_cond","median"),
          logcond_p95=("log10_dyn_cond", lambda s: s.quantile(0.95)),
          logcond_p99=("log10_dyn_cond", lambda s: s.quantile(0.99)),
          n=("log10_dyn_cond","size"),
      )
      .reset_index()
)

time_profile


In [ ]:
# roll_df must contain: method, split, field, horizon, mu_id, k0, rel_mean (or mse_fin, etc.)
intr_per_run = run_stats.copy()

merged = intr_per_run.merge(
    df_roll[["method","split","field","horizon","mu_id","k0","rel_mean","mse_fin"]],
    on=["method","split","field","horizon","mu_id","k0"],
    how="inner",
)

merged[["logcond_med","logcond_p99","cond_rate_1e5","laterr_med","rel_mean","mse_fin"]].corr()
